# Estudio de Convergencia (petición explícita del tutor)

**Objetivo del tutor**: no solo comparar accuracy final entre configuraciones, sino observar cómo evoluciona el entrenamiento época a época para cada arquitectura cuántica, cruzando tipo de entrelazador × n_layers × n_wires.

Preguntas que este estudio debe responder:
1. **Velocidad de convergencia**: ¿a partir de qué época se estabiliza la pérdida?
2. **Complejidad vs. aprendizaje**: ¿los circuitos más profundos/expresivos (StronglyEntanglingLayers con muchas capas) aprenden más rápido, o sufren *barren plateaus* (pérdida estancada desde época 1)?
3. **Sobreajuste**: ¿el modelo memoriza entrenamiento en épocas avanzadas sin mejorar en validación?

Este notebook reutiliza las mismas convenciones de columnas de `experiments_optimizado.csv` (tu pipeline principal), para que el registro de esta fase sea directamente compatible y combinable con el resto de tus resultados. Incluye protección contra el problema de **filas duplicadas por reinicios de kernel** detectado previamente: cada corrida se identifica de forma única y, si ya existe en el CSV, se sobrescribe en vez de duplicarse.

**Recomendación de ejecución**: este estudio tiene un número considerable de entrenamientos (ver Sección 5). Ejecuta este notebook convertido a `.py` dentro de una sesión `tmux` en el HPC, no directamente en la celda de Jupyter, para que un corte de VPN o de VS Code no mate el proceso a mitad de camino (ver última celda para el comando exacto).


## 1. Setup

In [ ]:
import os
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

import pennylane as qml


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


DEVICE = torch.device("cpu")
RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = RESULTS_DIR / "convergence_study.csv"

print(f"Dispositivo: {DEVICE}")
print("Nota: default.qubit de PennyLane simula en CPU; GPU no acelera la parte cuantica.")


## 2. Datos (mismo split que el pipeline principal)

In [ ]:
CONV_SEED = 42
BATCH_SIZE = 64
VAL_SPLIT = 0.1
DATA_DIR = "./data"

def get_dataloaders(seed=CONV_SEED, batch_size=BATCH_SIZE, val_split=VAL_SPLIT):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    full_train = torchvision.datasets.MNIST(root=DATA_DIR, train=True, download=True, transform=transform)
    test_set = torchvision.datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=transform)

    n_val = int(len(full_train) * val_split)
    n_train = len(full_train) - n_val
    generator = torch.Generator().manual_seed(seed)
    train_set, val_set = random_split(full_train, [n_train, n_val], generator=generator)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)
    print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")
    return train_loader, val_loader, test_loader


## 3. Circuito cuántico con forma de pesos dinámica

En vez de declarar `weight_shapes` a mano para cada combinación de `n_layers`/`n_wires` (como en el pipeline principal), usamos el método `.shape()` de cada plantilla para generar la forma correcta automáticamente — evita errores de dimensión al iterar sobre configuraciones variables. Esta es exactamente la práctica que indicó tu tutor.

`n_wires` se usa aquí como sinónimo de `n_qubits`, siguiendo la convención de la API de PennyLane para estas plantillas.


In [ ]:
def build_quantum_layer_dynamic(n_wires: int, n_layers: int, entangler: str, ranges=None):
    """Construye la capa cuantica usando .shape() para generar weight_shapes
    de forma dinamica y segura, en vez de calcularla a mano."""
    dev = qml.device("default.qubit", wires=n_wires)

    if entangler == "basic":
        shape = qml.BasicEntanglerLayers.shape(n_layers=n_layers, n_wires=n_wires)

        @qml.qnode(dev, interface="torch")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_wires))
            qml.BasicEntanglerLayers(weights, wires=range(n_wires))
            return [qml.expval(qml.PauliZ(i)) for i in range(n_wires)]

    elif entangler == "strong":
        shape = qml.StronglyEntanglingLayers.shape(n_layers=n_layers, n_wires=n_wires)

        @qml.qnode(dev, interface="torch")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_wires))
            if ranges is not None:
                qml.StronglyEntanglingLayers(weights, wires=range(n_wires), ranges=ranges)
            else:
                qml.StronglyEntanglingLayers(weights, wires=range(n_wires))
            return [qml.expval(qml.PauliZ(i)) for i in range(n_wires)]

    else:
        raise ValueError(f"entangler desconocido: {entangler}")

    weight_shapes = {"weights": shape}
    return qml.qnn.TorchLayer(circuit, weight_shapes)


## 4. Modelo (encoder + capa cuántica dinámica + clasificador)

In [ ]:
class SharedEncoder(nn.Module):
    def __init__(self, n_wires: int, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_wires),
            nn.Tanh(),
        )

    def forward(self, x):
        return self.net(x)


class ConvergenceModel(nn.Module):
    def __init__(self, n_wires: int, n_layers: int, entangler: str, ranges=None):
        super().__init__()
        self.encoder = SharedEncoder(n_wires)
        self.quantum_layer = build_quantum_layer_dynamic(n_wires, n_layers, entangler, ranges=ranges)
        self.classifier = nn.Linear(n_wires, 10)

    def forward(self, x):
        x = self.encoder(x) * torch.pi
        x = self.quantum_layer(x)
        return self.classifier(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## 5. Entrenamiento a épocas fijas (sin early stopping) + registro compatible con tu CSV principal

A diferencia del pipeline principal, aquí **no** se usa early stopping: el objetivo es observar la curva completa, incluyendo posible sobreajuste en épocas avanzadas, que el early stopping interrumpiría antes de que se manifieste. El registro de cada corrida usa las mismas columnas que tu `experiments_optimizado.csv` (run_name, seed, n_qubits, n_layers, entangler, encoding, encoder_hidden_dim, batch_size, learning_rate, epochs, test_accuracy, training_time_sec, n_trainable_params), añadiendo además el historial completo por época en un CSV separado (`convergence_study.csv`) para poder calcular los tres indicadores del tutor.


In [ ]:
def train_fixed_epochs(model, train_loader, val_loader, epochs: int, lr: float = 1e-3, run_name: str = ""):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)
        val_loss /= len(val_loader.dataset)
        val_acc = correct / total

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"[{run_name}] Epoca {epoch+1}/{epochs} | train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

    elapsed = time.time() - start_time
    return pd.DataFrame(history), elapsed


def append_or_update_csv(df_row: dict, csv_path: Path, key_col: str = "run_name"):
    """Guarda una fila en el CSV, sobrescribiendo si run_name ya existe.
    Evita el problema de filas duplicadas por reinicios de kernel detectado
    previamente en experiments_optimizado.csv."""
    new_row = pd.DataFrame([df_row])
    if csv_path.exists():
        existing = pd.read_csv(csv_path)
        existing = existing[existing[key_col] != df_row[key_col]]  # descarta version anterior si existe
        combined = pd.concat([existing, new_row], ignore_index=True)
    else:
        combined = new_row
    combined.to_csv(csv_path, index=False)


## 6. Diseño factorial del experimento

Cruce sistemático: **entangler** (basic, strong) × **n_layers** ∈ {1, 3, 5} × **n_wires** (rango ampliado respecto al barrido principal, para no repetir tal cual el tutorial).

**Idempotencia**: si el kernel se reinicia a mitad de la ejecución (como ya te pasó antes), puedes simplemente volver a correr esta celda — las combinaciones ya guardadas en `convergence_study.csv` se detectan y se saltan automáticamente, no se repiten ni se duplican.


In [ ]:
ENTANGLERS = ["basic", "strong"]
N_LAYERS_GRID = [1, 3, 5]
N_WIRES_GRID = [4, 6, 8, 10]   # ampliado respecto al barrido principal (4,6,8)
EPOCHS = 15                     # fijo, sin early stopping (tutor sugiere 10 o 15)
RANGES_TO_TEST = None           # None = patron por defecto de StronglyEntanglingLayers;
                                 # o fija p.ej. [1] para forzar un patron de alcance constante

total_runs = len(ENTANGLERS) * len(N_LAYERS_GRID) * len(N_WIRES_GRID)
print(f"Total de configuraciones: {total_runs}")


In [ ]:
train_loader, val_loader, _ = get_dataloaders()

# Cargar lo que ya esté guardado (si venimos de un reinicio de kernel)
if CSV_PATH.exists():
    combined = pd.read_csv(CSV_PATH)
    done_runs = set(combined["run_name"].unique()) if len(combined) else set()
else:
    combined = pd.DataFrame()
    done_runs = set()

print(f"Configuraciones ya completadas previamente: {len(done_runs)}")

all_histories = [combined] if len(combined) else []

for entangler in ENTANGLERS:
    for n_layers in N_LAYERS_GRID:
        for n_wires in N_WIRES_GRID:
            run_name = f"conv_{entangler}_l{n_layers}_w{n_wires}"

            if run_name in done_runs:
                print(f"[{run_name}] ya completado, saltando.")
                continue

            print(f"\n{'='*10} {run_name} {'='*10}")

            set_seed(CONV_SEED)
            model = ConvergenceModel(n_wires=n_wires, n_layers=n_layers, entangler=entangler, ranges=RANGES_TO_TEST)
            n_params = count_params(model)

            history_df, elapsed = train_fixed_epochs(model, train_loader, val_loader, epochs=EPOCHS, run_name=run_name)

            history_df["entangler"] = entangler
            history_df["n_layers"] = n_layers
            history_df["n_wires"] = n_wires
            history_df["n_qubits"] = n_wires  # alias, consistente con experiments_optimizado.csv
            history_df["run_name"] = run_name
            history_df["training_time_sec"] = round(elapsed, 1)
            history_df["n_trainable_params"] = n_params

            # Guardar INMEDIATAMENTE tras cada configuracion (no al final) para no perder
            # progreso si el kernel se cae a mitad del barrido
            all_histories.append(history_df)
            combined = pd.concat(all_histories, ignore_index=True)
            combined = combined.drop_duplicates(subset=["run_name", "epoch"], keep="last")
            combined.to_csv(CSV_PATH, index=False)

            print(f"[{run_name}] tiempo total={elapsed:.1f}s | parametros={n_params} | guardado en CSV")

print(f"\nCompletado. Total filas en {CSV_PATH}: {len(combined)}")


### 6.1 Curvas de val_loss por configuración, agrupadas por entangler

In [ ]:
combined = pd.read_csv(CSV_PATH)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, entangler in zip(axes, ENTANGLERS):
    subset = combined[combined["entangler"] == entangler]
    for (n_layers, n_wires), group in subset.groupby(["n_layers", "n_wires"]):
        group = group.sort_values("epoch")
        ax.plot(group["epoch"], group["val_loss"], marker="o", markersize=3,
                label=f"L={n_layers}, W={n_wires}")
    ax.set_title(f"Entangler: {entangler}")
    ax.set_xlabel("Epoca")
    ax.set_ylabel("Val loss")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "convergence_curves.png", dpi=150)
plt.show()


## 7. Indicadores operacionales: convergencia, barren plateau y overfitting

Para cada configuración se calculan tres indicadores a partir de su curva de `val_loss`/`train_loss`:

- **Época de convergencia**: primera época a partir de la cual el cambio de `val_loss` entre epocas consecutivas se mantiene por debajo de un umbral durante al menos `patience` epocas seguidas.
- **Barren plateau**: la perdida de entrenamiento apenas mejora desde la epoca 1 (mejora relativa menor al `rel_threshold`) — sospechoso de gradientes desvanecientes, especialmente esperable en `strong` con muchas capas.
- **Overfitting**: en las ultimas epocas, `train_loss` sigue bajando mientras `val_loss` sube.


In [ ]:
def find_convergence_epoch(val_losses, threshold=0.01, patience=2):
    """Primera epoca en la que |delta val_loss| < threshold durante `patience`
    epocas consecutivas. Devuelve None si nunca se estabiliza."""
    stable_count = 0
    for i in range(1, len(val_losses)):
        delta = abs(val_losses[i] - val_losses[i - 1])
        if delta < threshold:
            stable_count += 1
            if stable_count >= patience:
                return i - patience + 2
        else:
            stable_count = 0
    return None


def detect_barren_plateau(train_losses, rel_threshold=0.05):
    """True si la perdida de entrenamiento no mejora mas de un rel_threshold
    (5% por defecto) respecto a su valor inicial en toda la corrida."""
    initial = train_losses[0]
    final = train_losses[-1]
    improvement = (initial - final) / initial if initial != 0 else 0
    return improvement < rel_threshold


def detect_overfitting(train_losses, val_losses, tail=3):
    """True si, mirando las ultimas `tail` epocas, train_loss sigue bajando
    (tendencia negativa) mientras val_loss sube (tendencia positiva)."""
    if len(train_losses) < tail + 1:
        return False
    train_trend = train_losses[-1] - train_losses[-tail]
    val_trend = val_losses[-1] - val_losses[-tail]
    return train_trend < 0 and val_trend > 0


summary_rows = []
for (entangler, n_layers, n_wires), group in combined.groupby(["entangler", "n_layers", "n_wires"]):
    group = group.sort_values("epoch")
    train_losses = group["train_loss"].tolist()
    val_losses = group["val_loss"].tolist()

    summary_rows.append({
        "entangler": entangler,
        "n_layers": n_layers,
        "n_wires": n_wires,
        "convergence_epoch": find_convergence_epoch(val_losses),
        "barren_plateau": detect_barren_plateau(train_losses),
        "overfitting": detect_overfitting(train_losses, val_losses),
        "final_train_loss": round(train_losses[-1], 4),
        "final_val_loss": round(val_losses[-1], 4),
        "final_val_acc": round(group["val_acc"].iloc[-1], 4),
        "training_time_sec": round(group["training_time_sec"].iloc[0], 1),
        "n_trainable_params": group["n_trainable_params"].iloc[0],
    })

summary_df = pd.DataFrame(summary_rows).sort_values(["entangler", "n_layers", "n_wires"])
summary_df.to_csv(RESULTS_DIR / "convergence_summary.csv", index=False)
summary_df


## 8. Lectura de resultados para la memoria

Con `summary_df` ya se pueden responder directamente las tres preguntas del tutor:

1. **Velocidad de convergencia**: compara `convergence_epoch` entre `basic` y `strong`, y entre distintas profundidades — ¿los circuitos mas simples convergen antes de forma sistematica?
2. **Complejidad vs. aprendizaje / barren plateaus**: cruza `barren_plateau` con `entangler` y `n_layers` — ¿se concentra en `strong` con mas capas, como predice la literatura (McClean et al., 2018)?
3. **Sobreajuste**: cruza `overfitting` con la profundidad — ¿los circuitos de mayor capacidad memorizan antes en igualdad de epocas?


In [ ]:
pivot_convergence = summary_df.pivot_table(index="n_layers", columns="entangler", values="convergence_epoch")
pivot_barren = summary_df.pivot_table(index="n_layers", columns="entangler", values="barren_plateau", aggfunc="mean")
pivot_overfit = summary_df.pivot_table(index="n_layers", columns="entangler", values="overfitting", aggfunc="mean")

print("Epoca de convergencia promedio (por n_layers x entangler):")
display(pivot_convergence)
print("\nProporcion de configuraciones con barren plateau (por n_layers x entangler):")
display(pivot_barren)
print("\nProporcion de configuraciones con overfitting (por n_layers x entangler):")
display(pivot_overfit)


## 9. Cómo ejecutar esto de forma segura en el HPC (evitar el corte que ya te pasó)

1. Convierte este notebook a script: `jupyter nbconvert --to script estudio_convergencia_seccion16.ipynb`
2. En tu sesión SSH del HPC, abre una sesión `tmux` **antes** de lanzar el script:
```bash
tmux new -s convergencia
python estudio_convergencia_seccion16.py
# Ctrl+B, luego D para desconectar sin matar el proceso
# tmux attach -t convergencia   <- para volver a conectarte despues
```
3. Si aun asi el kernel se reinicia, simplemente vuelve a ejecutar el script completo: gracias a la comprobacion de `done_runs` en la Seccion 6, las configuraciones ya guardadas en `convergence_study.csv` se saltan automaticamente y no se duplican.
